In [ ]:
# =========================================================
# Imports et configuration
# =========================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os

import sys
import os
sys.path.append(os.path.abspath('..'))

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Import des variables partagées (suppose que config.py et data.py existent)
from config import RANDOM_STATE, TEST_SIZE, TARGET_REG, FEATURES_REG
from data import load_data, clean_data

sns.set_theme(style="whitegrid")

# =========================================================
# Chargement et préparation des données
# =========================================================
df = load_data()
df_clean = clean_data(df)

# Définition des variables explicatives et de la cible
X_reg = df_clean[FEATURES_REG]
y_reg = df_clean[TARGET_REG]

# Découpage Train / Test selon les conventions du groupe
X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg, y_reg, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

# =========================================================
# Modèle 1 : Régression Linéaire Simple (Loudness)
# =========================================================
# On utilise loudness car c'est généralement la variable la plus corrélée à l'énergie
X_train_simple = X_reg_train[['loudness']]
X_test_simple = X_reg_test[['loudness']]

model_reg_simple = LinearRegression()
model_reg_simple.fit(X_train_simple, y_reg_train)

# =========================================================
# Modèle 2 : Régression Linéaire Multiple
# =========================================================
model_reg_multi = LinearRegression()
model_reg_multi.fit(X_reg_train, y_reg_train)

# Prédictions du modèle multiple (retenu comme modèle principal)
y_reg_pred = model_reg_multi.predict(X_reg_test)

# =========================================================
# Évaluation et Comparaison (results_reg)
# =========================================================
def evaluate_model(name, model, X_test_data):
    pred = model.predict(X_test_data)
    return {
        'Modèle': name,
        'R²': round(r2_score(y_reg_test, pred), 3),
        'MAE': round(mean_absolute_error(y_reg_test, pred), 3),
        'RMSE': round(np.sqrt(mean_squared_error(y_reg_test, pred)), 3)
    }

results_reg = pd.DataFrame([
    evaluate_model("Régression Simple (Loudness)", model_reg_simple, X_test_simple),
    evaluate_model("Régression Multiple", model_reg_multi, X_reg_test)
])

print("--- Performances des modèles ---")
print(results_reg)

# =========================================================
# Interprétation des coefficients du modèle multiple
# =========================================================
print("\n--- Coefficients de la Régression Multiple ---")
coefficients = pd.DataFrame(
    list(zip(FEATURES_REG, model_reg_multi.coef_)), 
    columns=['Variable', 'Coefficient']
)
coefficients['Impact Absolu'] = coefficients['Coefficient'].abs()
coefficients = coefficients.sort_values(by='Impact Absolu', ascending=False).drop('Impact Absolu', axis=1)
print(coefficients)

# =========================================================
# Visualisation : Valeurs Réelles vs Prédites
# =========================================================
plt.figure(figsize=(8, 6))
plt.scatter(y_reg_test, y_reg_pred, alpha=0.3, color='royalblue')
plt.plot([0, 1], [0, 1], color='red', linestyle='--', linewidth=2) # Ligne parfaite y=x
plt.title('Régression Multiple : Énergie Réelle vs Prédite')
plt.xlabel('Énergie Réelle (y_reg_test)')
plt.ylabel('Énergie Prédite (y_reg_pred)')
plt.xlim(0, 1)
plt.ylim(0, 1)
plt.show()

# =========================================================
# Sauvegarde du modèle pour Streamlit
# =========================================================
os.makedirs('models', exist_ok=True)
joblib.dump(model_reg_multi, 'models/regression.joblib')
print("\nModèle sauvegardé dans 'models/regression.joblib'")

ModuleNotFoundError: No module named 'config'